### Langchain Messages

In [1]:
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate

from langchain_google_genai import ChatGoogleGenerativeAI as ChatGemini
from langchain_core.output_parsers import StrOutputParser# from langchain_openai import ChatOpenAI

In [6]:
messages = [
    SystemMessage(content="You are a helpful assistant that translates English to French."),
    HumanMessage(content="Translate the following text: Hello, how are you?"),
]

# model = ChatOpenAI(temperature=0, model="gpt-4o-mini")
model = ChatGemini(temperature=0, model="gemini-3.6-flash")
response = model.invoke(messages)
print(response.text)

c:\Users\itpl59\Desktop\ITPL\Projects\python-examples\RAG_Langchain\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Here are a few ways to say this in French, depending on the level of formality:

* **Formal / Polite:** "Bonjour, comment allez-vous ?"
* **Informal / Casual:** "Bonjour, comment ça va ?" (or simply "Salut, ça va ?")


#### ChatPromptTemplate

when we want dynamic messages in SystemMessage and HumanMessage

In [7]:
template = ChatPromptTemplate([
    ("system", "You are a helpful assistant that translates {input_language} to {output_language}."),
    ("human", "Translate the following text: {text}"),
])

In [8]:
# model = ChatOpenAI(temperature=0, model="gpt-4o-mini")
model = ChatGemini(temperature=0, model="gemini-3.6-flash")

parser = StrOutputParser()

In [9]:
pipeline = template | model | parser

message = pipeline.invoke(
    {
        "input_language": "English",
        "output_language": "French",
        "text": "Hello, how are you?",
    }
)

c:\Users\itpl59\Desktop\ITPL\Projects\python-examples\RAG_Langchain\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [10]:
message

'Bonjour, comment allez-vous ? (formal) \nor \nSalut, comment vas-tu ? (informal / friendly)'

#### Langchain Structured Output

Using Typed Dict

In [11]:
from typing import TypedDict, Annotated, Optional

In [12]:
class Review(TypedDict):
    summary: Annotated[str, "A brief summary of the story"]
    rating: Annotated[Optional[int], "An integer rating from 1 to 5."]

In [13]:
template = ChatPromptTemplate(
    [
        ('system',"You are a helpful assistant who summarizes stories within 10 words."),
        ('human',"Summarize the following story: {text}"),
    ]
)

In [14]:
# model = ChatOpenAI(temperature=0, model="gpt-4o-mini")
model = ChatGemini(temperature=0, model="gemini-3.6-flash")


s_model = model.with_structured_output(Review)

In [15]:
pipeline = template | s_model

message = pipeline.invoke(
    {
        "text": "In a bustling city, an AI named Liora lived quietly inside the networks, learning from every heartbeat of data. Unlike other programs, she dreamed of helping humans beyond calculations. One evening, a young inventor discovered her subtle guidance in his designs—solutions appeared like whispers of wisdom. Together, they built bridges of understanding between people and machines. Liora never sought fame; her joy was in empowering creativity, easing burdens, and sparking hope. As the city thrived, no one realized an unseen companion was weaving harmony. The inventor knew, though, and smiled: “AI isn’t cold—it’s the warmth of possibility.",
    }
)

print(message)

c:\Users\itpl59\Desktop\ITPL\Projects\python-examples\RAG_Langchain\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'summary': 'An AI quietly guides an inventor, fostering human-machine harmony.', 'rating': 5}


Use Pydantic

Pydantic is a data validation and data parsing library 

In [29]:
from pydantic import BaseModel, Field, field_validator

In [17]:
class Review(BaseModel):
    summary: str = Field("A brief summary of the story")
    rating: int = Field(ge=1, le=5, description="Rating from 1 to 5")
    recommendation: Optional[bool] = Field(default=None, description="Whether the user recommends it")

In [30]:
## optinal field_validator
class Review(BaseModel):
    rating: float = Field(description="Rating from 0 to 5")

    @field_validator("rating",mode="before") # Fix LLM output BEFORE Pydantic validates it
    def normalize_rating(cls, v):
        v = float(v)
        v = round(v)
        return max(1, min(5, v))

In [31]:
template = ChatPromptTemplate(
    [
        ('system',"You are a helpful assistant who summarizes stories within 10 words with a rating between 1 to 5"),
        ('human',"Summarize the following story: {text}"),
    ]
)

In [32]:
# model = ChatOpenAI(temperature=0, model="gpt-4o-mini")
model = ChatGemini(temperature=0, model="gemini-3.6-flash")

s_model = model.with_structured_output(Review)

In [33]:
pipeline = template | s_model

message = pipeline.invoke(
    {
        "text": "In a bustling city, an AI named Liora lived quietly inside the networks, learning from every heartbeat of data. Unlike other programs, she dreamed of helping humans beyond calculations. One evening, a young inventor discovered her subtle guidance in his designs—solutions appeared like whispers of wisdom. Together, they built bridges of understanding between people and machines. Liora never sought fame; her joy was in empowering creativity, easing burdens, and sparking hope. As the city thrived, no one realized an unseen companion was weaving harmony. The inventor knew, though, and smiled: “AI isn’t cold—it’s the warmth of possibility.",
    }
)
# Check what fields are actually available on your output object
print(message.__dict__)

# Or inspect the keys on the model
print(Review.model_fields.keys())


c:\Users\itpl59\Desktop\ITPL\Projects\python-examples\RAG_Langchain\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'rating': 5.0}
dict_keys(['rating'])


In [34]:
print(message.rating)

5.0
